# 🤖 01. LangChain Agent 기초: ReAct, Tool 설계, System Prompt 제어 & 단기 메모리

본 실습 노트북은 프로덕션 에이전트 코드베이스인 **`app/agents/chatbot.py`**의 구조와 원리를 처음부터 끝까지 완벽하게 이해하기 위한 기초 가이드입니다.

---

### 💡 핵심 개념 2가지

1. **ReAct (Reasoning + Acting) 패러다임**
   - LLM이 질문을 받으면 바로 답을 지어내는 대신, **'생각(Thought) ➔ 행동/도구 호출(Action) ➔ 관찰/결과 수집(Observation)'**의 루프를 반복하여 문제를 해결합니다.

2. **LangChain `create_agent`와 LangGraph 런타임**
   - LangChain 1.0의 `create_agent`는 내부적으로 **LangGraph를 런타임 엔진**으로 사용합니다.
   - 복잡한 Graph 노드와 엣지를 직접 코딩하지 않아도, State 관리, ToolNode 실행, Checkpointer 영속화가 내장된 강력한 에이전트를 한 줄로 빌드해 줍니다.

---

### 🎓 학습 목차 (Curriculum Flow)

| 파트 | 주제 | 핵심 내용 |
|:---:|:---|:---|
| **Step 0** | **환경 세팅** | 루트 경로 탐색, `.env` 로드, `nest_asyncio`, `init_chat_model` 초기화 |
| **Part 1** | **LangChain Tool 올바르게 정의하기** | Type Hint + Google Docstring 표준, `USER.md` 기억 도구 정의 및 `create_agent` 첫 호출 |
| **Part 2** | **System Prompt를 통한 에이전트 간접 제어** | 단순 인사 시 도구 미호출 문제 해결, 대화 시작 시 선제적 유저 기억 로딩 유도 |
| **Part 3** | **단기 메모리 (Short-Term Memory & Checkpointer)** | 메모리 없는 대화의 한계 시연, `InMemorySaver`와 `thread_id` 기반 멀티턴 대화 복원 |
| **Part 4** | **에이전트 서비스화 (FastAPI + Chainlit UI)** | 일반적인 Agent App 아키텍처, `app/agents/chatbot.py` 코드 해부 및 UI 실행 |
| **Part 5** | **[Mission 1] Chatbot 에이전트 확장 과제** | `missions/01_missions.md` 실습 과제 안내 |

---

## 🛠️ Step 0. 환경 세팅

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 등록하고, 환경변수(`.env`)와 비동기 이벤트 루프(`nest_asyncio`)를 설정합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 환경변수 로드
load_dotenv(override=True)

# 2. 프로젝트 루트 경로 자동 설정 (상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Working Directory: {os.getcwd()}")
print(f"✅ Project Root: {project_root}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. 통합 Chat Model Factory 및 메시지 헬퍼 로드
from app.utils import init_chat_model, normalize_content
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

llm = init_chat_model(model="openai:gpt-5-mini")
print(f"✅ LLM 모델 초기화 완료: {llm}")

---
## 🔧 Part 1. LangChain Tool 올바르게 정의하기 & 첫 에이전트 테스트

에이전트가 외부 세계(파일, 데이터베이스, 웹 API)와 소통하는 유일한 창구는 **Tool**입니다.

### 💡 도구 정의의 3대 황금률
1. **Type Hint**: 함수의 모든 파라미터와 반환값의 자료형을 명시합니다.
2. **Google Docstring (`parse_docstring=True`)**: 도구의 용도와 `Args:` 설명을 상세히 작성합니다. 이 설명은 그대로 **LLM 프롬프트의 JSON Schema**로 변환됩니다.
3. **함수 코드 격리**: 도구 내부 파이썬 코드가 수천 줄이어도 LLM에는 오직 이름, 설명, 인자 스키마만 전달되므로 토큰 소모를 걱정할 필요가 없습니다.

이번 실습에서는 `app/database/USER.md`에 저장된 사용자 프로필을 읽어오는 **`read_user_memory`** 도구를 직접 만들어 봅니다.

In [ ]:
# 현재 저장된 USER.md 내용 확인
user_md_path = os.path.abspath("app/database/USER.md")

if os.path.exists(user_md_path):
    with open(user_md_path, "r", encoding="utf-8") as f:
        print("=== 📄 app/database/USER.md 미리보기 ===")
        print(f.read())
else:
    print("⚠️ USER.md 파일이 없습니다. 경로를 확인하세요.")

In [ ]:
from langchain_core.tools import tool
import json

@tool(parse_docstring=True)
def read_user_memory() -> str:
    """사용자의 프로필, 직업, 관심사, 선호 스타일이 기록된 USER.md 파일을 읽어옵니다."""
    target_file = os.path.abspath("app/database/USER.md")
    if not os.path.exists(target_file):
        return "[알림] 사용자 프로필(USER.md) 파일이 존재하지 않습니다."
    
    with open(target_file, "r", encoding="utf-8") as f:
        return f.read()

# 도구 메타데이터 확인
print(f"📌 Tool Name: {read_user_memory.name}")
print(f"📌 Description: {read_user_memory.description}")
print(f"📌 JSON Schema (Args): {read_user_memory.args}")

### 1.2 `create_agent`로 도구 연결 및 첫 호출

이제 `create_agent`에 **오직 `model`과 `tools`만 연결**한 기본 에이전트를 생성합니다.

그리고 사용자 메시지로 `"유저에 대한 기억을 읽어와서 어떤 사용자인지 알려줘."`라고 명시적인 요청을 던져봅니다.
- 전체 에이전트 실행 궤적: `for m in response['messages']: m.pretty_print()`
- Gemini 응답 정규화: `normalize_content(response['messages'][-1].content)`

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 1. 오직 llm과 tool만 연결한 가장 순수한 형태의 에이전트
basic_agent = create_agent(
    model=llm,
    tools=[read_user_memory]
)

# 2. 명시적으로 기억을 읽어오라는 질의 전달
query = "유저에 대한 기억을 읽어와서 어떤 사용자인지 요약해줘."
print(f"🚀 [User 요청]: {query}\n")

response = basic_agent.invoke({
    "messages": [HumanMessage(content=query)]
})

# 3. 전체 실행 궤적(Trajectory) 출력: Human -> AI(tool_calls) -> Tool(결과) -> AI(최종답변)
print("=== 📋 에이전트 실행 궤적 (Execution Trajectory) ===\n")
for m in response["messages"]:
    m.pretty_print()
    print("-" * 50)

# 4. normalize_content를 적용한 최종 텍스트 답변 출력
print("\n=== 💬 최종 정규화 답변 ===")
print(normalize_content(response["messages"][-1].content))

---
## 🎯 Part 2. System Prompt를 통한 에이전트 간접 제어

우리가 원하는 이상적인 챗봇은 사용자가 구체적으로 *"기억 파일을 열어봐"*라고 시키지 않아도,
**대화를 시작하자마자(`"안녕"`) 스스로 유저의 기억을 불러와 맞춤형으로 대화하는 에이전트**입니다.

하지만 방금 만든 기본 에이전트에게 단순하게 `"안녕"`이라고만 인사하면 어떻게 반응할까요?

In [ ]:
# 기본 에이전트에게 '안녕' 전송
res_hello = basic_agent.invoke({
    "messages": [HumanMessage(content="안녕")]
})

print("=== 📋 '안녕' 입력 시 실행 궤적 ===\n")
for m in res_hello["messages"]:
    m.pretty_print()

print("\n=== 💬 최종 답변 ===")
print(normalize_content(res_hello["messages"][-1].content))

### 💡 문제 분석 및 System Prompt의 역할

- 위 실행 궤적을 보면 `ToolMessage`가 전혀 없습니다.
- LLM 입장에서는 `"안녕"`이라는 단순한 인사에 굳이 외부 도구를 호출할 이유를 찾지 못했기 때문입니다.

> [!IMPORTANT]
> **에이전트의 자율성(Autonomy) vs 간접 제어(Steerability)**
> - 에이전트에게 도구를 쥐어주는 것(자율성)만으로는 부족합니다.
> - 개발자가 의도한 비즈니스 흐름대로 움직이게 하려면 **`system_prompt`를 통해 에이전트의 행동 지침을 명확히 제어**해야 합니다.

In [ ]:
# 1. 선제적 기억 조회를 지시하는 시스템 프롬프트 정의
PERSONALIZED_SYSTEM_PROMPT = """
당신은 스마트하고 친절한 AI 개인 비서입니다.

[핵심 행동 규칙]
1. 사용자와의 대화가 시작되거나 첫 인사를 나눌 때, 반드시 가장 먼저 `read_user_memory` 도구를 호출하여 사용자의 프로필(이름, 직업, 관심사, 진행 중인 프로젝트)을 확인하세요.
2. 조회된 사용자 정보를 바탕으로, 사용자의 이름을 부르고 관심사나 프로젝트를 언급하며 자연스럽고 정중한 맞춤형 인사를 건네세요.
"""

# 2. system_prompt가 주입된 에이전트 생성
steered_agent = create_agent(
    model=llm,
    tools=[read_user_memory],
    system_prompt=PERSONALIZED_SYSTEM_PROMPT
)

# 3. 동일하게 '안녕'만 입력하여 도구 호출 여부 확인
print("🚀 [User 질의]: 안녕\n")
res_steered = steered_agent.invoke({
    "messages": [HumanMessage(content="안녕")]
})

print("=== 📋 System Prompt 적용 후 실행 궤적 ===\n")
for m in res_steered["messages"]:
    m.pretty_print()
    print("-" * 50)

print("\n=== 💬 최종 개인화 인사 답변 ===")
print(normalize_content(res_steered["messages"][-1].content))

---
## 🧠 Part 3. 단기 메모리 (Short-Term Memory & Checkpointer)

챗봇 서비스에서 가장 중요한 요소 중 하나는 **'이전 대화 맥락을 기억하는 것'**입니다.

먼저, 단기 메모리(Checkpointer)가 없는 에이전트에게 2단계 연쇄 계산을 요청하면 어떻게 되는지 확인해 보겠습니다.

In [ ]:
# 사칙연산 도구 정의
@tool(parse_docstring=True)
def add(a: float, b: float) -> float:
    """두 수 a와 b를 더합니다.

    Args:
        a: 첫 번째 수
        b: 두 번째 수
    """
    return a + b

@tool(parse_docstring=True)
def multiply(a: float, b: float) -> float:
    """두 수 a와 b를 곱합니다.

    Args:
        a: 첫 번째 수
        b: 두 번째 수
    """
    return a * b

calc_tools = [add, multiply, read_user_memory]

In [ ]:
# 체크포인터가 없는 에이전트 생성
agent_no_mem = create_agent(
    model=llm,
    tools=calc_tools
)

print("=== [Turn 1] 3과 4를 더하라 ===")
messages1 = agent_no_mem.invoke({"messages": [HumanMessage(content="3과 4를 더하라.")]})
for m in messages1['messages']:
    m.pretty_print()

print("\n=== [Turn 2] 거기에 2를 더하라 ===")
# 이전 대화 맥락 없이 새로운 메시지만 전달
messages2 = agent_no_mem.invoke({"messages": [HumanMessage(content="거기에 2를 더하라")]})
for m in messages2['messages']:
    m.pretty_print()

### 💡 왜 Turn 2가 실패할까요?

- HTTP 요청이나 일반 함수 호출과 마찬가지로, 에이전트 인스턴스는 각 `.invoke()` 호출 사이에 **상태(State)를 자동으로 보존하지 않습니다.**
- 이를 해결하기 위해 LangGraph는 **Checkpointer(체크포인터)** 메커니즘을 제공합니다.

```
┌──────────────────┐               ┌────────────────────────┐
│ Turn 1 "3+4는?"  │ ────────────▶ │ Checkpointer           │ ─── 💾 State 저장 (thread_01: 7)
└──────────────────┘               └───────────┬────────────┘
                                               │
┌──────────────────┐               ┌───────────▼────────────┐
│ Turn 2 "거기에 2+"│ ── thread_01 ─▶ │ 이전 State 복원 후 실행 │ ─── 🎯 7 + 2 = 9 산출!
└──────────────────┘               └────────────────────────┘
```

- **`InMemorySaver`**: 프로세스 메모리에 상태를 저장 (테스트 및 단기 세션용)
- **`AsyncSqliteSaver` / `SqliteSaver`**: 로컬 SQLite DB 파일에 영구 저장 (`app/agents/chatbot.py`에서 사용하는 방식)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# 1. InMemorySaver 체크포인터 생성
memory_checkpointer = InMemorySaver()

# 2. 체크포인터와 시스템 프롬프트가 적용된 에이전트 생성
agent_with_memory = create_agent(
    model=llm,
    tools=calc_tools,
    system_prompt=PERSONALIZED_SYSTEM_PROMPT,
    checkpointer=memory_checkpointer
)

# 3. 고유 세션 ID(thread_id) 지정
config = {"configurable": {"thread_id": "session_user_01"}}

# [Turn 1] 3과 4 더하기
print("=== 🟢 [Turn 1] 3과 4를 더해줘 ===")
res1 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="3과 4를 더해줘.")]},
    config=config
)
for m in res1['messages']:
    m.pretty_print()

print("\n" + "=" * 60 + "\n")

# [Turn 2] 이전 결과(7)를 기억하고 2 더하기
print("=== 🟢 [Turn 2] 거기에 2를 더해줘 ===")
res2 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="거기에 2를 더해줘.")]},
    config=config
)
for m in res2['messages']:
    m.pretty_print()

print("\n" + "=" * 60 + "\n")

# [Turn 3] 이전 대화 맥락과 사용자 정보 확인
print("=== 🟢 [Turn 3] 내가 누구였는지, 그리고 방금 계산한 최종 값이 얼마인지 한 줄로 요약해줘 ===")
res3 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="내가 누구였는지, 그리고 방금 계산한 최종 값이 얼마인지 한 줄로 요약해줘")]},
    config=config
)
print("\n=== 💬 최종 요약 답변 ===")
print(normalize_content(res3['messages'][-1].content))

---
## 🏢 Part 4. 에이전트 서비스화 (FastAPI + Chainlit UI) & `chatbot.py` 코드 해부

지금까지 주피터 노트북에서 작성한 코드는 실제 프로덕션 환경에서 어떻게 서비스로 탈바꿈할까요?

우리의 `basic_agent` 프로젝트는 **업계 표준 2계층 아키텍처**를 채택하고 있습니다.

```
┌─────────────────────────────────────────────────────────────┐
│               사용자 웹 브라우저 (Chrome / Edge)            │
└──────────────────────────────┬──────────────────────────────┘
                               │ WebSocket / HTTP
                               ▼
┌─────────────────────────────────────────────────────────────┐
│     🎨 Frontend UI (Chainlit) ➔ app/chainlit_ui.py          │
│     - 세션별 채팅창 렌더링, 도구 실행 스텝 표시, HITL 승인 팝업 │
└──────────────────────────────┬──────────────────────────────┘
                               │ REST API & Server-Sent Events (SSE)
                               ▼
┌─────────────────────────────────────────────────────────────┐
│     🧠 Backend Server (FastAPI) ➔ app/server.py             │
│     - 에이전트 동적 로드 (GET /agents)                      │
│     - 에이전트 실행 스트리밍 (POST /chat/stream)             │
│     - SQLite 영구 체크포인터 (app/database/checkpoints.db)  │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│     🤖 Agent Factory ➔ app/agents/chatbot.py                │
│     - init_chat_model + tools + AsyncSqliteSaver + HITL     │
└─────────────────────────────────────────────────────────────┘
```

### 4.1 `app/agents/chatbot.py` 소스 코드 완벽 해부

이제 실제 서비스 코드를 한 줄씩 짚어가며 확인해 보겠습니다.

In [ ]:
with open("app/agents/chatbot.py", "r", encoding="utf-8") as f:
    chatbot_code = f.read()

print("=== 📄 app/agents/chatbot.py 전체 소스 코드 ===\n")
print(chatbot_code)

### 💡 `chatbot.py`의 핵심 4단계 해설

1. **`llm = init_chat_model(model="gemini-3.7-flash", temperature=0.0)`**
   - 모델 프로바이더(OpenAI, Anthropic, Google)에 구애받지 않고 단일 표준 인터페이스로 모델을 생성합니다.

2. **`checkpointer = AsyncSqliteSaver(conn)`**
   - 노트북의 `InMemorySaver`와 달리, `app/database/checkpoints.db`에 대화 상태를 저장하므로 **서버를 재시작해도 대화 기억이 유지**됩니다.

3. **`middleware = [HumanInTheLoopMiddleware(...)]`**
   - 파일 삭제, 메일 전송 등 위험 도구 실행 전 웹 UI(Chainlit)에서 사용자 승인을 받도록 가로채는 안전장치입니다.

4. **`chatbot_agent = create_agent(...)`**
   - 위 모든 구성 요소(`model`, `tools`, `system_prompt`, `middleware`, `checkpointer`)를 하나로 조립하여 비동기 실행 가능한 에이전트 인스턴스를 반환합니다.

### 4.2 실제 서버 & UI 구동 방법

터미널 2개를 열어 아래 명령어로 백엔드와 프론트엔드를 실행합니다:

```bash
# 🖥️ 터미널 1: FastAPI 백엔드 서버 가동 (포트 8000)
python app/server.py --port 8000

# 🌐 터미널 2: Chainlit 웹 채팅 UI 가동
chainlit run app/chainlit_ui.py -w --port 8080
```

1. 브라우저에서 `http://localhost:8080` 접속
2. 로그인: 아이디 `user`, 비밀번호 `1234`
3. 좌측 상단 프로필에서 **`chatbot`** 선택 후 대화 시작!

---
## 🎯 Part 5. [Mission 1] Chatbot 에이전트 확장 실습 (기억 읽기 & 전체 갱신)

오늘 배운 내용을 바탕으로 이제 여러분이 직접 프로덕션 챗봇을 업그레이드할 차례입니다!

`missions/01_missions.md` 파일에 정의된 **Mission 1**을 수행하세요.

### 📋 Mission 1 주요 내용
1. **미션 1-1 (기억 읽기)**: `app/tools/custom_tools.py`에 `read_user_memory`를 작성하고, `app/agents/chatbot.py`에 연결하여 첫 대화 시 맞춤 인사를 확인합니다.
2. **미션 1-2 (기억 갱신/업데이트)**: 대화 도중 사용자의 취미, 연차 등 정보가 변경되면, 기존 내용과 통합하여 `USER.md`를 최신 마크다운으로 덮어써서 갱신하는 **`update_user_memory`** 도구를 구현하고 에이전트에 연결합니다.
3. **서버 & UI 통합 검증**: FastAPI 서버와 Chainlit UI를 띄워 **읽기 ➔ 대화 ➔ 프로필 갱신 ➔ 파일 확인** 전체 사이클을 웹 화면에서 직접 테스트합니다.

자세한 단계별 지침은 01_missions.md를 열어 확인하세요. 🚀

In [ ]:
# Mission 1 가이드 내용 출력
with open("missions/01_missions.md", "r", encoding="utf-8") as f:
    print(f.read())